# Urdu-Punjabi XLM-RoBERTa Training on Google Colab
Train the model with FREE GPU in 5-10 minutes!

**Steps:**
1. Upload `combined_training_dataset.json` to Colab
2. Run all cells
3. Download trained model
4. Deploy to HuggingFace

In [ ]:
# Step 1: Install dependencies
!pip install torch transformers datasets scikit-learn huggingface_hub accelerate -q

In [ ]:
# Step 2: Upload dataset
from google.colab import files
import os

print("Upload expanded_dataset_2500.json from ml_model/dataset/ folder")
uploaded = files.upload()

# Create directories
os.makedirs('dataset', exist_ok=True)
os.makedirs('training_data', exist_ok=True)

# Move uploaded file
!mv expanded_dataset_2500.json dataset/

In [ ]:
# Step 3: Training configuration
import torch
import json
import numpy as np
from pathlib import Path
from datetime import datetime
from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    XLMRobertaConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Configuration - EXPANDED DATASET
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
BATCH_SIZE = 32  # Increased for GPU
EPOCHS = 3  # Increased for larger dataset
LEARNING_RATE = 2e-5
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01

In [ ]:
# Step 4: Load and prepare data
with open('dataset/expanded_dataset_2500.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print(f"Dataset loaded: {dataset['metadata']['name']}")
print(f"Total entries: {dataset['metadata']['total_entries']}")

# Prepare training pairs
training_pairs = []

# Process Urdu training data
for vocab in dataset["training_data"]["urdu_train"]:
    word = vocab.get("word", "")
    translation = vocab.get("translation", "")
    example = vocab.get("example", "")
    example_translation = vocab.get("example_translation", "")
    
    # Exact match (score 1.0)
    training_pairs.append({
        "text_a": word,
        "text_b": word,
        "label": 1.0
    })
    
    # Translation (score 0.3)
    if translation:
        training_pairs.append({
            "text_a": word,
            "text_b": translation,
            "label": 0.3
        })
    
    # Example sentences (exact match score 1.0)
    if example:
        training_pairs.append({
            "text_a": example,
            "text_b": example,
            "label": 1.0
        })
        
        # Sentence translation (score 0.4)
        if example_translation:
            training_pairs.append({
                "text_a": example,
                "text_b": example_translation,
                "label": 0.4
            })
    
    # Wrong answer (score 0.0)
    training_pairs.append({
        "text_a": word,
        "text_b": "غلط جواب",
        "label": 0.0
    })

# Process Punjabi training data
for vocab in dataset["training_data"]["punjabi_train"]:
    word = vocab.get("word", "")
    translation = vocab.get("translation", "")
    example = vocab.get("example", "")
    example_translation = vocab.get("example_translation", "")
    
    # Exact match
    training_pairs.append({
        "text_a": word,
        "text_b": word,
        "label": 1.0
    })
    
    # Translation
    if translation:
        training_pairs.append({
            "text_a": word,
            "text_b": translation,
            "label": 0.3
        })
    
    # Example sentences
    if example:
        training_pairs.append({
            "text_a": example,
            "text_b": example,
            "label": 1.0
        })
        
        if example_translation:
            training_pairs.append({
                "text_a": example,
                "text_b": example_translation,
                "label": 0.4
            })
    
    # Wrong answer
    training_pairs.append({
        "text_a": word,
        "text_b": "غلط",
        "label": 0.0
    })

print(f"Total training pairs generated: {len(training_pairs)}")

# Use predefined test/validation sets
test_pairs = []
for vocab in dataset["training_data"]["test"]:
    word = vocab.get("word", "")
    test_pairs.append({
        "text_a": word,
        "text_b": word,
        "label": 1.0
    })

val_pairs = []
for vocab in dataset["training_data"]["validation"]:
    word = vocab.get("word", "")
    val_pairs.append({
        "text_a": word,
        "text_b": word,
        "label": 1.0
    })

train_data = training_pairs
val_data = val_pairs

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_pairs)}")

In [ ]:
# Step 5: Tokenize data
tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_NAME)

def create_dataset(data):
    texts_a = [d["text_a"] for d in data]
    texts_b = [d["text_b"] for d in data]
    labels = [d["label"] for d in data]
    
    encodings = tokenizer(
        texts_a,
        texts_b,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="np"
    )
    
    return Dataset.from_dict({
        "input_ids": encodings["input_ids"].tolist(),
        "attention_mask": encodings["attention_mask"].tolist(),
        "labels": labels
    })

train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
print("Datasets created!")

In [ ]:
# Step 6: Initialize model
config = XLMRobertaConfig.from_pretrained(MODEL_NAME)
config.num_labels = 1
config.problem_type = "regression"

model = XLMRobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config
)
model.to(device)
print("Model loaded!")

In [ ]:
# Step 7: Training arguments
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if len(predictions.shape) > 1:
        predictions = predictions.squeeze()
    predictions = np.clip(predictions, 0, 1)
    
    mse = mean_squared_error(labels, predictions)
    mae = mean_absolute_error(labels, predictions)
    accuracy = np.mean(np.abs(predictions - labels) <= 0.2)
    
    return {"mse": mse, "mae": mae, "accuracy": accuracy}

training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mse",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Trainer initialized!")

In [ ]:
# Step 8: TRAIN! (5-10 minutes on GPU)
print("Starting training...")
trainer.train()
print("\nTraining complete!")

In [ ]:
# Step 9: Save model
output_dir = "./trained_model_v3_2500"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

# Save config
config_info = {
    "model_name": MODEL_NAME,
    "model_version": "v3",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "training_samples": len(train_data),
    "validation_samples": len(val_data),
    "trained_at": datetime.now().isoformat(),
    "languages": ["urdu", "punjabi", "english"],
    "dataset": "expanded_2500_entries_pakistani_muslim",
    "total_vocab": 2500,
    "urdu_train": 1050,
    "punjabi_train": 1050,
    "test": 200,
    "validation": 200,
    "notes": "Pakistani Muslim oriented - Islamic content, no Sikh greetings"
}

with open(f"{output_dir}/training_config.json", 'w') as f:
    json.dump(config_info, f, indent=2)

print(f"Model saved to {output_dir}")
print(f"Training pairs: {len(train_data)}")
print(f"Validation pairs: {len(val_data)}")

In [ ]:
# Step 10: Test the model
model.eval()

test_pairs = [
    ("Ø®ÙˆØ´", "Ø®ÙˆØ´"),           # Exact match
    ("Ù¾ÛŒØ§Ø±", "Ù¾ÛŒØ§Ø±"),         # Exact match
    ("Ø³Ù„Ø§Ù…", "Ø§Ù„Ø³Ù„Ø§Ù…"),        # Partial
    ("Ø´Ú©Ø±ÛŒÛ", "thank you"),   # Translation
    ("Ú©ØªØ§Ø¨", "Ù‚Ù„Ù…"),          # Wrong
]

print("\nTesting model:")
for expected, user_input in test_pairs:
    inputs = tokenizer(expected, user_input, truncation=True, 
                      padding="max_length", max_length=MAX_LENGTH, 
                      return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        score = torch.sigmoid(outputs.logits).item()
    
    print(f"Expected: '{expected}' | User: '{user_input}' | Score: {score:.3f}")

In [ ]:
# Step 11: Download trained model
import os
from google.colab import files

# Check if model folder exists
if os.path.exists('trained_model_v3_2500'):
    print("Creating zip file...")
    !zip -r trained_model_v3_2500.zip trained_model_v3_2500/
    
    if os.path.exists('trained_model_v3_2500.zip'):
        print("Downloading trained model (2500 entries)...")
        files.download('trained_model_v3_2500.zip')
        print("✅ Download complete!")
    else:
        print("❌ Error: Zip file not created")
else:
    print("❌ Error: Model folder 'trained_model_v3_2500' not found")
    print("Make sure Step 9 (Save model) completed successfully")

In [ ]:
# Step 12: (Optional) Deploy to HuggingFace
from huggingface_hub import login, HfApi
import os

# REPLACE WITH YOUR VALID TOKEN FROM: https://huggingface.co/settings/tokens
HF_TOKEN = "YOUR_NEW_TOKEN_HERE"  # Get from huggingface.co/settings/tokens (Write permission)
HF_REPO_ID = "RAFAY-484/Urdu-Punjabi-V2"

if not os.path.exists('trained_model_v3_2500'):
    print("❌ Error: Model folder not found. Run Step 9 first.")
else:
    print("Logging in to HuggingFace...")
    login(token=HF_TOKEN)

    api = HfApi()
    print(f"Uploading 2500-entry model to {HF_REPO_ID}...")
    api.upload_folder(
        folder_path="./trained_model_v3_2500",
        repo_id=HF_REPO_ID,
        token=HF_TOKEN,
        commit_message="Updated with 2500 entries - Pakistani Muslim oriented"
    )

    print(f"\n✅ Deployed to https://huggingface.co/{HF_REPO_ID}")
    print(f"API: https://api-inference.huggingface.co/models/{HF_REPO_ID}")
    print(f"Model now trained on 2500 words/sentences (1050 Urdu + 1050 Punjabi + 400 test/val)")